# 06 — Relational Graph Convolutional Network (`rgcn_leakage_safe_v2`)

This notebook trains a leakage-controlled **relational graph convolutional network (R-GCN)** over the frozen point-event dataset.

It uses:

- the fused point-event feature bank and primary event-to-flow edges from Notebook 03;
- the frozen chronological split and `model_target` from Notebook 04;
- the frozen temporal Transformer reference from Notebook 05.

The graph is causal and directed:

- earlier site events may send messages to later site events;
- earlier observations of the same aircraft may send messages to later observations;
- earlier events in the same episode may send messages to later episode events;
- sampled NF3 flow nodes send messages to point-event nodes;
- event nodes never send messages back into shared flow nodes.

The final comparison evaluates:

- **H2:** R-GCN test AUPRC minus Transformer test AUPRC is at least `0.03`;
- **H3:** full lag/relation-aware R-GCN test AUPRC minus the lag/relation ablation is at least `0.05`, while full-model p95 local inference latency is at most `500 ms`.

> The aircraft and NF3 records are controlled cross-domain pairings. Model performance does not establish a real site-coincident or causal aircraft–cyber relationship.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from numbers import Integral, Real
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence, cast

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import torch
    from torch import nn
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook 06 requires PyTorch. Install it in the active Jupyter environment "
        "before running this notebook. PyTorch Geometric is not required."
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# -----------------------------------------------------------------------------
# Reproducibility and resource configuration.
# Environment variables allow smaller smoke tests without editing the notebook.
# -----------------------------------------------------------------------------
RANDOM_STATE = int(os.environ.get("RGCN_RANDOM_STATE", "42"))
MODEL_VERSION = "rgcn_leakage_safe_v2"
PRIMARY_VARIANT = "full_lag_relation"
ABLATION_VARIANT = "lag_relation_ablation"
EVENT_ONLY_VARIANT = "event_only_self_loop"  # v3 ablation: no event-event or flow-event relations, self-loop only (= MLP on the event node)
INCLUDE_EVENT_ONLY_VARIANT = os.environ.get("RGCN_EVENT_ONLY_VARIANT", "0") == "1"

MAX_FLOWS_PER_EVENT = int(os.environ.get("RGCN_MAX_FLOWS_PER_EVENT", "16"))
EDGE_READ_CHUNK_ROWS = int(os.environ.get("RGCN_EDGE_CHUNK_ROWS", "250000"))
LAG_WEIGHT_SCALE_S = float(os.environ.get("RGCN_LAG_WEIGHT_SCALE_S", "120"))

HIDDEN_DIM = int(os.environ.get("RGCN_HIDDEN_DIM", "64"))
N_LAYERS = int(os.environ.get("RGCN_N_LAYERS", "2"))
DROPOUT = float(os.environ.get("RGCN_DROPOUT", "0.15"))
LEARNING_RATE = float(os.environ.get("RGCN_LEARNING_RATE", "0.001"))
WEIGHT_DECAY = float(os.environ.get("RGCN_WEIGHT_DECAY", "0.0001"))
MAX_EPOCHS = int(os.environ.get("RGCN_MAX_EPOCHS", "50"))
EARLY_STOPPING_PATIENCE = int(os.environ.get("RGCN_PATIENCE", "8"))
GRADIENT_CLIP_NORM = float(os.environ.get("RGCN_GRADIENT_CLIP_NORM", "1.0"))

ATTRIBUTION_SAMPLE_MAX = int(os.environ.get("RGCN_ATTRIBUTION_SAMPLE_MAX", "128"))
LOCAL_LATENCY_SAMPLE_MAX = int(os.environ.get("RGCN_LATENCY_SAMPLE_MAX", "100"))
LOCAL_LATENCY_WARMUP_RUNS = int(os.environ.get("RGCN_LATENCY_WARMUP_RUNS", "5"))
FULL_GRAPH_LATENCY_RUNS = int(os.environ.get("RGCN_FULL_GRAPH_LATENCY_RUNS", "20"))

H2_REQUIRED_AUPRC_MARGIN = 0.03
H3_REQUIRED_AUPRC_GAIN = 0.05
H3_MAX_P95_LATENCY_MS = 500.0

if MAX_FLOWS_PER_EVENT < 1:
    raise ValueError("MAX_FLOWS_PER_EVENT must be at least 1.")
if EDGE_READ_CHUNK_ROWS < 1:
    raise ValueError("EDGE_READ_CHUNK_ROWS must be at least 1.")
if N_LAYERS < 1:
    raise ValueError("N_LAYERS must be at least 1.")
if HIDDEN_DIM < 4:
    raise ValueError("HIDDEN_DIM must be at least 4.")

PROJECT_ROOT = Path.cwd()
BASE_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2"
ALIGNMENT_DIR = BASE_DIR / "nf3_alignment_v2"
MODELING_DIR = BASE_DIR / "modeling_v4"
TRANSFORMER_DIR = BASE_DIR / "transformer_v2"

FEATURES_IN = ALIGNMENT_DIR / "candidate_point_events_v2_with_context_and_nf3_primary.csv"
EDGES_IN = ALIGNMENT_DIR / "event_flow_edges_primary_bounded_v2.csv"
SPLITS_IN = MODELING_DIR / "model_split_assignments_v4.csv"
FEATURE_MANIFEST_IN = MODELING_DIR / "feature_manifest_v4.json"
TARGET_DEFINITION_IN = MODELING_DIR / "model_target_definition_v4.json"
TRANSFORMER_REFERENCE_IN = TRANSFORMER_DIR / "transformer_reference_v2.json"

OUTPUT_DIR = BASE_DIR / os.environ.get("RGCN_OUTPUT_SUBDIR", "rgcn_leakage_safe_v2")  # v3: keep original outputs intact
MODELS_DIR = OUTPUT_DIR / "models"
PREPROCESSING_DIR = OUTPUT_DIR / "preprocessing"
FIGURES_DIR = OUTPUT_DIR / "figures"

SELECTED_EDGES_OUT = OUTPUT_DIR / "selected_event_flow_edges_v2.csv"
NODE_MANIFEST_OUT = OUTPUT_DIR / "rgcn_node_manifest_v2.csv"
GRAPH_RELATIONS_OUT = OUTPUT_DIR / "rgcn_relation_statistics_v2.csv"
RESULTS_OUT = OUTPUT_DIR / "rgcn_results_v2.csv"
PREDICTIONS_OUT = OUTPUT_DIR / "rgcn_predictions_v2.csv"
THRESHOLDS_OUT = OUTPUT_DIR / "rgcn_thresholds_v2.json"
HISTORY_OUT = OUTPUT_DIR / "rgcn_training_history_v2.csv"
LATENCY_OUT = OUTPUT_DIR / "rgcn_latency_v2.csv"
RELATION_NORMS_OUT = OUTPUT_DIR / "rgcn_relation_weight_norms_v2.csv"
FEATURE_ATTRIBUTION_OUT = OUTPUT_DIR / "rgcn_feature_attribution_v2.csv"
COMPARISON_OUT = OUTPUT_DIR / "rgcn_vs_transformer_v2.csv"
HYPOTHESIS_OUT = OUTPUT_DIR / "rgcn_hypothesis_evaluation_v2.csv"
GRAPH_MANIFEST_OUT = OUTPUT_DIR / "rgcn_graph_manifest_v2.json"
REFERENCE_OUT = OUTPUT_DIR / "rgcn_reference_v2.json"
CONFIG_OUT = OUTPUT_DIR / "rgcn_config_v2.json"
SUMMARY_OUT = OUTPUT_DIR / "rgcn_build_summary_v2.json"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "rgcn_data_dictionary_v2.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

for directory in [OUTPUT_DIR, MODELS_DIR, PREPROCESSING_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Project root:       {PROJECT_ROOT}")
print(f"Feature input:      {FEATURES_IN}")
print(f"Edge input:         {EDGES_IN}")
print(f"PyTorch device:     {DEVICE}")
print(f"Output directory:   {OUTPUT_DIR}")


## 1. Validation, conversion, preprocessing, and metric helpers

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def require_text(value: object, field_name: str) -> str:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, bytes):
        text = value.decode("utf-8", errors="strict")
    else:
        text = str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")
    return text


def require_int(value: object, field_name: str) -> int:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (Integral, np.integer)):
        return int(value)
    if isinstance(value, (Real, np.floating)):
        number = float(value)
        if not math.isfinite(number) or not number.is_integer():
            raise ValueError(f"{field_name} must be a finite whole number; received {value!r}.")
        return int(number)
    text = require_text(value, field_name)
    try:
        number = float(text)
    except ValueError as exc:
        raise ValueError(f"{field_name} must be integer-compatible; received {value!r}.") from exc
    if not math.isfinite(number) or not number.is_integer():
        raise ValueError(f"{field_name} must be a finite whole number; received {value!r}.")
    return int(number)


def require_float(value: object, field_name: str) -> float:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, (Real, np.number)):
        number = float(value)
    else:
        text = require_text(value, field_name)
        try:
            number = float(text)
        except ValueError as exc:
            raise ValueError(f"{field_name} must be numeric; received {value!r}.") from exc
    if not math.isfinite(number):
        raise ValueError(f"{field_name} must be finite; received {value!r}.")
    return number


def optional_float(value: object) -> float | None:
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    try:
        number = float(str(value))
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    text = require_text(value, field_name)
    try:
        timestamp = pd.Timestamp(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{field_name} is not a valid timestamp: {value!r}."
        ) from exc

    if pd.isna(timestamp):
        raise ValueError(f"{field_name} is not a valid timestamp: {value!r}.")

    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")

    return timestamp


def slugify(value: object) -> str:
    text = require_text(value, "slug value").lower()
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def has_both_classes(values: pd.Series) -> bool:
    numeric = pd.to_numeric(values, errors="coerce").dropna().astype("int8")
    return set(numeric.unique().tolist()) == {0, 1}


def make_dense_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def transformed_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    try:
        return [str(value) for value in preprocessor.get_feature_names_out().tolist()]
    except Exception:
        return []


def safe_roc_auc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    if np.unique(y_true).size < 2:
        return float("nan")
    return float(roc_auc_score(y_true, probabilities))


def safe_auprc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    if int(np.sum(y_true)) == 0:
        return 0.0
    return float(average_precision_score(y_true, probabilities))


def select_f1_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, float]:
    candidates = np.unique(np.concatenate(([0.0, 0.5, 1.0], probabilities.astype(float))))
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in candidates:
        predicted = (probabilities >= threshold).astype(np.int8)
        score = float(f1_score(y_true, predicted, zero_division=0))
        if score > best_f1 or (math.isclose(score, best_f1) and threshold > best_threshold):
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def classification_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, float | int]:
    predicted = (probabilities >= threshold).astype(np.int8)
    matrix = confusion_matrix(y_true, predicted, labels=[0, 1])
    tn, fp, fn, tp = [int(value) for value in matrix.ravel()]
    return {
        "auprc": safe_auprc(y_true, probabilities),
        "roc_auc": safe_roc_auc(y_true, probabilities),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def existing_unique(columns: Iterable[object], frame: pd.DataFrame) -> list[str]:
    output: list[str] = []
    seen: set[str] = set()
    for value in columns:
        column = str(value)
        if column in frame.columns and column not in seen:
            output.append(column)
            seen.add(column)
    return output


def usable_columns_for_training(
    training_frame: pd.DataFrame,
    numeric_candidates: list[str],
    categorical_candidates: list[str],
) -> tuple[list[str], list[str], list[str]]:
    usable_numeric: list[str] = []
    usable_categorical: list[str] = []
    dropped: list[str] = []
    for column in numeric_candidates:
        numeric = pd.to_numeric(training_frame[column], errors="coerce")
        if numeric.notna().sum() == 0 or numeric.nunique(dropna=True) <= 1:
            dropped.append(column)
        else:
            usable_numeric.append(column)
    for column in categorical_candidates:
        categorical = training_frame[column].astype("string")
        if categorical.notna().sum() == 0 or categorical.nunique(dropna=True) <= 1:
            dropped.append(column)
        else:
            usable_categorical.append(column)
    return usable_numeric, usable_categorical, dropped


def build_preprocessor(
    numeric_columns: list[str],
    categorical_columns: list[str],
) -> ColumnTransformer:
    transformers: list[tuple[str, Pipeline, list[str]]] = []
    if numeric_columns:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )
    if categorical_columns:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_dense_one_hot_encoder()),
                    ]
                ),
                categorical_columns,
            )
        )
    if not transformers:
        raise ValueError("No usable predictors remain for preprocessing.")
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


def as_float32_matrix(value: object, expected_rows: int, name: str) -> np.ndarray:
    matrix = np.asarray(value, dtype=np.float32)
    if matrix.ndim != 2 or matrix.shape[0] != expected_rows:
        raise ValueError(f"Unexpected {name} matrix shape: {matrix.shape}.")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} matrix contains non-finite values.")
    return matrix


seed_everything(RANDOM_STATE)

## 2. Load and validate the frozen Notebook 03–05 artifacts

In [ ]:
required_paths = [
    FEATURES_IN,
    EDGES_IN,
    SPLITS_IN,
    FEATURE_MANIFEST_IN,
    TARGET_DEFINITION_IN,
    TRANSFORMER_REFERENCE_IN,
]
for required_path in required_paths:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required upstream artifact was not found: {required_path}\n"
            "Run the corrected Notebooks 03, 04, and 05 successfully from the project root first."
        )

features = pd.read_csv(FEATURES_IN, low_memory=False)
splits = pd.read_csv(SPLITS_IN, low_memory=False)
feature_manifest = json.loads(FEATURE_MANIFEST_IN.read_text(encoding="utf-8"))
target_definition = json.loads(TARGET_DEFINITION_IN.read_text(encoding="utf-8"))
transformer_reference = json.loads(TRANSFORMER_REFERENCE_IN.read_text(encoding="utf-8"))

ID_COLUMN = "event_id"
TIME_COLUMN = "aircraft_event_time_utc"
GROUP_COLUMN = "aircraft_key"
EPISODE_COLUMN = "episode_id"
TARGET_COLUMN = "model_target"
SPLIT_COLUMN = "chronological_split"

required_feature_columns = {ID_COLUMN, TIME_COLUMN, GROUP_COLUMN}
required_split_columns = {ID_COLUMN, TARGET_COLUMN, SPLIT_COLUMN}
missing_feature_columns = sorted(required_feature_columns - set(features.columns))
missing_split_columns = sorted(required_split_columns - set(splits.columns))
if missing_feature_columns:
    raise KeyError(f"Feature bank is missing required columns: {missing_feature_columns}")
if missing_split_columns:
    raise KeyError(f"Split assignment table is missing required columns: {missing_split_columns}")
if features[ID_COLUMN].duplicated().any():
    raise ValueError("Feature bank contains duplicate event_id values.")
if splits[ID_COLUMN].duplicated().any():
    raise ValueError("Split assignment table contains duplicate event_id values.")

split_columns = [ID_COLUMN, TARGET_COLUMN, SPLIT_COLUMN]
for optional_column in ["cyber_attack_window_label", "aircraft_group_split"]:
    if optional_column in splits.columns:
        split_columns.append(optional_column)

model_data = features.merge(
    splits[split_columns],
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)
if len(model_data) != len(features) or len(model_data) != len(splits):
    raise ValueError(
        "Feature and split rows do not match one-to-one: "
        f"features={len(features):,}, splits={len(splits):,}, merged={len(model_data):,}."
    )

model_data[ID_COLUMN] = model_data[ID_COLUMN].astype("string")
model_data[GROUP_COLUMN] = model_data[GROUP_COLUMN].astype("string")
model_data[TIME_COLUMN] = model_data[TIME_COLUMN].map(
    lambda value: require_utc_timestamp(value, TIME_COLUMN)
)
model_data[TARGET_COLUMN] = pd.to_numeric(model_data[TARGET_COLUMN], errors="raise").astype("int8")
model_data[SPLIT_COLUMN] = model_data[SPLIT_COLUMN].astype("string")
if EPISODE_COLUMN in model_data.columns:
    model_data[EPISODE_COLUMN] = model_data[EPISODE_COLUMN].astype("string")

required_nonmissing = [ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, TARGET_COLUMN, SPLIT_COLUMN]
if model_data[required_nonmissing].isna().any().any():
    raise ValueError("Required identifiers, timestamps, targets, or split values contain missing data.")
if not set(model_data[TARGET_COLUMN].unique().tolist()).issubset({0, 1}):
    raise ValueError("model_target must contain only 0 and 1.")
if set(model_data[SPLIT_COLUMN].unique().tolist()) != {"train", "validation", "test"}:
    raise ValueError(
        "chronological_split must contain exactly train, validation, and test. "
        f"Observed: {sorted(model_data[SPLIT_COLUMN].unique().tolist())}"
    )

model_data = model_data.sort_values([TIME_COLUMN, ID_COLUMN], kind="mergesort").reset_index(drop=True)
split_order = model_data[SPLIT_COLUMN].map({"train": 0, "validation": 1, "test": 2})
if split_order.isna().any() or not split_order.is_monotonic_increasing:
    raise ValueError("Chronological split assignments are not monotonic after event-time sorting.")
if not has_both_classes(model_data.loc[model_data[SPLIT_COLUMN].eq("train"), TARGET_COLUMN]):
    raise ValueError("The chronological training partition must contain both target classes.")

manifest_feature_sets = feature_manifest.get("feature_sets", {})
if "fused" not in manifest_feature_sets:
    raise KeyError("feature_manifest_v4.json does not contain the fused feature set.")

transformer_test_auprc: float = require_float(
    transformer_reference.get("test_auprc"),
    "Transformer test AUPRC",
)

print(f"Point events:           {len(model_data):,}")
print(f"Unique aircraft keys:  {model_data[GROUP_COLUMN].nunique():,}")
print(f"Positive targets:      {require_int(model_data[TARGET_COLUMN].sum(), 'positive targets'):,}")
print(f"Chronological splits:  {model_data[SPLIT_COLUMN].value_counts().to_dict()}")
print(f"Transformer test AUPRC:{transformer_test_auprc: .6f}")

## 3. Select a label-blind, bounded event-to-flow subgraph

The Notebook 03 edge table can contain millions of rows. This notebook keeps at most `MAX_FLOWS_PER_EVENT` flows for each point event, selected only by:

1. smallest absolute flow-start lag;
2. largest interval overlap;
3. smallest source-row number;
4. deterministic `flow_id` ordering.

No NF3 label or attack category is read or used during graph construction.

In [ ]:
EDGE_REQUIRED_COLUMNS = {
    "event_id",
    "flow_id",
    "relation_type",
    "signed_flow_start_lag_s",
    "absolute_flow_start_lag_s",
    "overlap_seconds",
    "flow_start_utc",
    "flow_end_utc",
}
EDGE_OPTIONAL_COLUMNS = [
    "source_row_number",
    "protocol",
    "src_port",
    "dst_port",
    "window_name",
]


def select_bounded_event_flow_edges(
    edge_path: Path,
    valid_event_ids: set[str],
    maximum_per_event: int,
    chunk_rows: int,
) -> pd.DataFrame:
    header = pd.read_csv(edge_path, nrows=0)
    missing = sorted(EDGE_REQUIRED_COLUMNS - set(header.columns))
    if missing:
        raise KeyError(f"Event-flow edge table is missing required columns: {missing}")

    use_columns = sorted(EDGE_REQUIRED_COLUMNS) + [
        column for column in EDGE_OPTIONAL_COLUMNS if column in header.columns
    ]
    candidate_chunks: list[pd.DataFrame] = []
    source_default = np.iinfo(np.int64).max

    for chunk_number, chunk in enumerate(
        pd.read_csv(edge_path, usecols=use_columns, chunksize=chunk_rows, low_memory=False),
        start=1,
    ):
        chunk["event_id"] = chunk["event_id"].astype("string")
        chunk["flow_id"] = chunk["flow_id"].astype("string")
        chunk = chunk.loc[chunk["event_id"].isin(valid_event_ids)].copy()
        if chunk.empty:
            continue

        for column in [
            "signed_flow_start_lag_s",
            "absolute_flow_start_lag_s",
            "overlap_seconds",
            "src_port",
            "dst_port",
        ]:
            if column in chunk.columns:
                chunk[column] = pd.to_numeric(chunk[column], errors="coerce")

        chunk["absolute_flow_start_lag_s"] = chunk[
            "absolute_flow_start_lag_s"
        ].fillna(chunk["signed_flow_start_lag_s"].abs())
        chunk["overlap_seconds"] = chunk["overlap_seconds"].fillna(0.0).clip(lower=0.0)
        if "source_row_number" in chunk.columns:
            chunk["_source_sort"] = pd.to_numeric(
                chunk["source_row_number"], errors="coerce"
            ).fillna(source_default)
        else:
            chunk["_source_sort"] = source_default

        chunk = chunk.dropna(
            subset=["event_id", "flow_id", "absolute_flow_start_lag_s"]
        )
        chunk = chunk.sort_values(
            [
                "event_id",
                "absolute_flow_start_lag_s",
                "overlap_seconds",
                "_source_sort",
                "flow_id",
            ],
            ascending=[True, True, False, True, True],
            kind="mergesort",
        )
        candidate_chunks.append(
            chunk.groupby("event_id", sort=False, group_keys=False).head(maximum_per_event)
        )

        if chunk_number % 10 == 0:
            print(f"Read {chunk_number:,} edge chunks...")

    if not candidate_chunks:
        raise ValueError("No event-to-flow edges matched the point-event identifiers.")

    candidates = pd.concat(candidate_chunks, ignore_index=True)
    candidates = candidates.sort_values(
        [
            "event_id",
            "absolute_flow_start_lag_s",
            "overlap_seconds",
            "_source_sort",
            "flow_id",
        ],
        ascending=[True, True, False, True, True],
        kind="mergesort",
    )
    candidates = candidates.drop_duplicates(["event_id", "flow_id"], keep="first")
    selected = (
        candidates.groupby("event_id", sort=False, group_keys=False)
        .head(maximum_per_event)
        .reset_index(drop=True)
    )
    return selected.drop(columns=["_source_sort"], errors="ignore")


valid_event_ids = set(model_data[ID_COLUMN].astype(str).tolist())
selected_edges = select_bounded_event_flow_edges(
    EDGES_IN,
    valid_event_ids,
    MAX_FLOWS_PER_EVENT,
    EDGE_READ_CHUNK_ROWS,
)

selected_edges["event_id"] = selected_edges["event_id"].astype("string")
selected_edges["flow_id"] = selected_edges["flow_id"].astype("string")
selected_edges["relation_type"] = (
    selected_edges["relation_type"].astype("string").str.strip().str.lower()
)
selected_edges["flow_start_utc"] = selected_edges["flow_start_utc"].map(
    lambda value: require_utc_timestamp(value, "flow_start_utc")
)
selected_edges["flow_end_utc"] = selected_edges["flow_end_utc"].map(
    lambda value: require_utc_timestamp(value, "flow_end_utc")
)
selected_edges["signed_flow_start_lag_s"] = pd.to_numeric(
    selected_edges["signed_flow_start_lag_s"], errors="raise"
)
selected_edges["absolute_flow_start_lag_s"] = pd.to_numeric(
    selected_edges["absolute_flow_start_lag_s"], errors="raise"
)
selected_edges["overlap_seconds"] = pd.to_numeric(
    selected_edges["overlap_seconds"], errors="raise"
).clip(lower=0.0)

if selected_edges[["event_id", "flow_id", "relation_type"]].isna().any().any():
    raise ValueError("Selected event-flow edges contain missing identifiers or relation types.")
if selected_edges.duplicated(["event_id", "flow_id"]).any():
    raise ValueError("Selected event-flow edge table contains duplicate event-flow pairs.")
if selected_edges.groupby("event_id").size().max() > MAX_FLOWS_PER_EVENT:
    raise AssertionError("The maximum-flows-per-event bound was not enforced.")

selected_edges.to_csv(SELECTED_EDGES_OUT, index=False)

edge_event_count = int(selected_edges["event_id"].nunique())
print(f"Selected event-flow edges: {len(selected_edges):,}")
print(f"Events with selected flows:{edge_event_count: ,}")
print(f"Unique selected flows:     {selected_edges['flow_id'].nunique():,}")
print("Relation distribution:")
display(selected_edges["relation_type"].value_counts().rename_axis("relation_type").to_frame("edges"))

## 4. Fit training-only event and flow preprocessing

In [ ]:
train_event_mask = model_data[SPLIT_COLUMN].eq("train")

# Event features use the same frozen fused candidate set as Notebooks 04 and 05.
fused_specification = manifest_feature_sets["fused"]
event_numeric_candidates = existing_unique(
    fused_specification.get("numeric", []), model_data
)
event_categorical_candidates = existing_unique(
    fused_specification.get("categorical", []), model_data
)

excluded_fields = set(feature_manifest.get("excluded_target_derived_columns", []))
excluded_fields.update(feature_manifest.get("excluded_identifier_and_provenance_columns", []))
excluded_fields.update(feature_manifest.get("excluded_alignment_audit_columns", []))
excluded_fields.update({ID_COLUMN, TIME_COLUMN, GROUP_COLUMN, TARGET_COLUMN, SPLIT_COLUMN})
forbidden_event_features = sorted(
    set(event_numeric_candidates + event_categorical_candidates) & excluded_fields
)
if forbidden_event_features:
    raise AssertionError(
        f"Forbidden fields entered the R-GCN event feature set: {forbidden_event_features}"
    )

for column in event_numeric_candidates:
    model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
for column in event_categorical_candidates:
    model_data[column] = model_data[column].astype("string")

event_numeric, event_categorical, dropped_event_features = usable_columns_for_training(
    model_data.loc[train_event_mask],
    event_numeric_candidates,
    event_categorical_candidates,
)
event_raw_columns = event_numeric + event_categorical
if not event_raw_columns:
    raise ValueError("No usable fused event features remain for the R-GCN.")

event_preprocessor = build_preprocessor(event_numeric, event_categorical)
event_preprocessor.fit(model_data.loc[train_event_mask, event_raw_columns])
event_matrix = as_float32_matrix(
    event_preprocessor.transform(model_data[event_raw_columns]),
    len(model_data),
    "event feature",
)
event_feature_names = transformed_feature_names(event_preprocessor)
if len(event_feature_names) != event_matrix.shape[1]:
    event_feature_names = [
        f"event_transformed_feature_{index:04d}"
        for index in range(event_matrix.shape[1])
    ]
joblib.dump(event_preprocessor, PREPROCESSING_DIR / "fused_event_preprocessor.joblib")

# Flow-node features are static, label-free fields from the selected edge table.
flow_table = (
    selected_edges.sort_values(["flow_id", "flow_start_utc"], kind="mergesort")
    .drop_duplicates("flow_id", keep="first")
    .reset_index(drop=True)
)
flow_table["flow_duration_s"] = (
    flow_table["flow_end_utc"] - flow_table["flow_start_utc"]
).dt.total_seconds().clip(lower=0.0)
for column in ["src_port", "dst_port"]:
    if column not in flow_table.columns:
        flow_table[column] = np.nan
    flow_table[column] = pd.to_numeric(flow_table[column], errors="coerce")
if "protocol" not in flow_table.columns:
    flow_table["protocol"] = "unknown"
flow_table["protocol"] = flow_table["protocol"].astype("string").fillna("unknown")

flow_numeric_candidates = ["flow_duration_s", "src_port", "dst_port"]
flow_categorical_candidates = ["protocol"]

train_event_ids = set(model_data.loc[train_event_mask, ID_COLUMN].astype(str).tolist())
train_flow_ids = set(
    selected_edges.loc[
        selected_edges["event_id"].astype(str).isin(train_event_ids), "flow_id"
    ].astype(str).tolist()
)
train_flow_mask = flow_table["flow_id"].astype(str).isin(train_flow_ids)
if not train_flow_mask.any():
    raise ValueError(
        "No selected flow nodes are adjacent to chronological training events. "
        "The flow preprocessor cannot be fit without leaking validation/test information."
    )

flow_numeric, flow_categorical, dropped_flow_features = usable_columns_for_training(
    flow_table.loc[train_flow_mask],
    flow_numeric_candidates,
    flow_categorical_candidates,
)
flow_raw_columns = flow_numeric + flow_categorical
if not flow_raw_columns:
    raise ValueError("No usable label-free flow-node features remain.")

flow_preprocessor = build_preprocessor(flow_numeric, flow_categorical)
flow_preprocessor.fit(flow_table.loc[train_flow_mask, flow_raw_columns])
flow_matrix = as_float32_matrix(
    flow_preprocessor.transform(flow_table[flow_raw_columns]),
    len(flow_table),
    "flow feature",
)
flow_feature_names = transformed_feature_names(flow_preprocessor)
if len(flow_feature_names) != flow_matrix.shape[1]:
    flow_feature_names = [
        f"flow_transformed_feature_{index:04d}"
        for index in range(flow_matrix.shape[1])
    ]
joblib.dump(flow_preprocessor, PREPROCESSING_DIR / "flow_preprocessor.joblib")

# Stable global node indexing: all event nodes, then all selected flow nodes.
event_count = len(model_data)
flow_count = len(flow_table)
event_node_index = {
    event_id: index
    for index, event_id in enumerate(model_data[ID_COLUMN].astype(str).tolist())
}
flow_node_index = {
    flow_id: event_count + index
    for index, flow_id in enumerate(flow_table["flow_id"].astype(str).tolist())
}

node_manifest_event = pd.DataFrame(
    {
        "node_index": np.arange(event_count, dtype=np.int64),
        "node_type": "event",
        "node_id": model_data[ID_COLUMN].astype(str),
        "split": model_data[SPLIT_COLUMN].astype(str),
        "target": model_data[TARGET_COLUMN].astype("int8"),
        "event_time_utc": model_data[TIME_COLUMN],
    }
)
node_manifest_flow = pd.DataFrame(
    {
        "node_index": np.arange(event_count, event_count + flow_count, dtype=np.int64),
        "node_type": "flow",
        "node_id": flow_table["flow_id"].astype(str),
        "split": pd.NA,
        "target": pd.NA,
        "event_time_utc": pd.NaT,
    }
)
node_manifest = pd.concat([node_manifest_event, node_manifest_flow], ignore_index=True)
node_manifest.to_csv(NODE_MANIFEST_OUT, index=False)

print(f"Event feature shape: {event_matrix.shape}")
print(f"Flow feature shape:  {flow_matrix.shape}")
print(f"Event nodes:         {event_count:,}")
print(f"Flow nodes:          {flow_count:,}")
print(f"Total graph nodes:   {len(node_manifest):,}")
print(f"Dropped event fields:{len(dropped_event_features):,}")
print(f"Dropped flow fields: {len(dropped_flow_features):,}")

## 5. Construct the causal relational graphs

The full graph uses three flow-to-event relations (`pre_event`, `overlaps_event_time`, and `post_event`) and lag/overlap edge weights. The ablation collapses those three relations into one unweighted `flow_to_event` relation. Both variants retain the same causal event-event structure.

In [ ]:
@dataclass(frozen=True)
class RelationEdges:
    source: np.ndarray
    destination: np.ndarray
    weight: np.ndarray


@dataclass(frozen=True)
class GraphBundle:
    name: str
    relations: dict[str, RelationEdges]
    event_count: int
    flow_count: int


def add_relation(
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]],
    name: str,
    source: Sequence[int] | np.ndarray,
    destination: Sequence[int] | np.ndarray,
    weight: Sequence[float] | np.ndarray,
) -> None:
    source_array = np.asarray(source, dtype=np.int64)
    destination_array = np.asarray(destination, dtype=np.int64)
    weight_array = np.asarray(weight, dtype=np.float32)
    if not (len(source_array) == len(destination_array) == len(weight_array)):
        raise ValueError(f"Relation {name!r} has inconsistent edge-array lengths.")
    if len(source_array) == 0:
        return
    storage.setdefault(name, []).append((source_array, destination_array, weight_array))


def finalize_relations(
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]]
) -> dict[str, RelationEdges]:
    finalized: dict[str, RelationEdges] = {}
    for name, parts in storage.items():
        source = np.concatenate([part[0] for part in parts]).astype(np.int64, copy=False)
        destination = np.concatenate([part[1] for part in parts]).astype(np.int64, copy=False)
        weight = np.concatenate([part[2] for part in parts]).astype(np.float32, copy=False)
        if not np.isfinite(weight).all():
            raise ValueError(f"Relation {name!r} contains non-finite edge weights.")
        finalized[name] = RelationEdges(source=source, destination=destination, weight=weight)
    if not finalized:
        raise ValueError("The graph contains no relations.")
    return finalized


def normalize_flow_relation(value: object) -> str:
    text = str(value).strip().lower()
    mapping = {
        "pre_event": "flow_pre_event_to_event",
        "overlaps_event_time": "flow_overlaps_event_to_event",
        "post_event": "flow_post_event_to_event",
    }
    return mapping.get(text, "flow_other_to_event")


def prior_index_edges(group_values: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    positions = pd.Series(np.arange(len(group_values), dtype=np.int64), index=group_values.index)
    previous = positions.groupby(group_values.astype("string"), sort=False).shift(1)
    valid = previous.notna()
    source = previous.loc[valid].astype("int64").to_numpy()
    destination = positions.loc[valid].astype("int64").to_numpy()
    return source, destination


def build_graph_bundle(variant: str) -> GraphBundle:
    storage: dict[str, list[tuple[np.ndarray, np.ndarray, np.ndarray]]] = {}
    if variant == EVENT_ONLY_VARIANT:
        # v3 ablation: the only relation is an event self-loop, so message passing carries no
        # information from other events or from flows; the model reduces to an MLP on the event node.
        loop = np.arange(event_count, dtype=np.int64)
        add_relation(storage, "event_self", loop, loop, np.ones(event_count, dtype=np.float32))
        return GraphBundle(name=variant, relations=finalize_relations(storage), event_count=event_count, flow_count=flow_count)

    # Global causal site stream: immediate prior event -> current event.
    if event_count > 1:
        add_relation(
            storage,
            "site_previous_event",
            np.arange(event_count - 1, dtype=np.int64),
            np.arange(1, event_count, dtype=np.int64),
            np.ones(event_count - 1, dtype=np.float32),
        )

    # Causal previous observation of the same stable aircraft.
    aircraft_source, aircraft_destination = prior_index_edges(model_data[GROUP_COLUMN])
    add_relation(
        storage,
        "aircraft_previous_event",
        aircraft_source,
        aircraft_destination,
        np.ones(len(aircraft_source), dtype=np.float32),
    )

    # Causal previous event in the same episode, when episode IDs are available.
    if EPISODE_COLUMN in model_data.columns and model_data[EPISODE_COLUMN].notna().any():
        episode_source, episode_destination = prior_index_edges(model_data[EPISODE_COLUMN])
        add_relation(
            storage,
            "episode_previous_event",
            episode_source,
            episode_destination,
            np.ones(len(episode_source), dtype=np.float32),
        )

    selected = selected_edges.copy()
    selected["event_node"] = selected["event_id"].astype(str).map(event_node_index)
    selected["flow_node"] = selected["flow_id"].astype(str).map(flow_node_index)
    if selected[["event_node", "flow_node"]].isna().any().any():
        raise ValueError("At least one selected edge could not be mapped to a graph node.")

    absolute_lag = pd.to_numeric(selected["absolute_flow_start_lag_s"], errors="raise").to_numpy(float)
    overlap = pd.to_numeric(selected["overlap_seconds"], errors="raise").to_numpy(float)
    proximity_weight = np.exp(-absolute_lag / max(LAG_WEIGHT_SCALE_S, 1e-6))
    # v3.2: the overlap-weight scale is derived from TRAINING-partition edges only (a global maximum over all
    # selected edges used evaluation-partition information). RGCN_OVERLAP_SCALE_SOURCE=all restores the old behaviour.
    # v3.3: the training mask comes from the EVENT's partition in the frozen split table (the selected-edge frame
    # carries no partition column, so v3.2's check on selected.columns never fired and fell back to the global max).
    _scale_source = os.environ.get("RGCN_OVERLAP_SCALE_SOURCE", "train")
    _train_events = set(model_data.loc[model_data[SPLIT_COLUMN].astype(str).eq("train"), "event_id"].astype(str))
    _train_mask = selected["event_id"].astype(str).isin(_train_events).to_numpy()
    _global_scale = max(float(np.nanmax(overlap)), 1.0)
    _train_scale = max(float(np.nanmax(overlap[_train_mask])) if _train_mask.any() else 1.0, 1.0)
    overlap_scale = _train_scale if _scale_source == "train" else _global_scale
    if _scale_source == "train" and not _train_mask.any():
        raise ValueError("No training-partition edges found for the overlap scale.")
    overlap_weight = np.clip(0.5 + 0.5 * np.log1p(overlap) / np.log1p(overlap_scale), 0.0, 1.0)
    (OUTPUT_DIR / "overlap_scale_v33.json").write_text(json.dumps({
        "scale_source": _scale_source, "overlap_scale_used_s": overlap_scale, "training_max_overlap_s": _train_scale,
        "global_max_overlap_s": _global_scale, "training_edges": int(_train_mask.sum()), "all_edges": int(len(overlap)),
        "max_weight_change_vs_global": float(np.nanmax(np.abs(np.clip(0.5 + 0.5 * np.log1p(overlap) / np.log1p(overlap_scale), 0, 1) - (0.5 + 0.5 * np.log1p(overlap) / np.log1p(_global_scale))))),
    }, indent=2))
    print(f"overlap scale source={_scale_source}: train max {_train_scale:.3f} s, global max {_global_scale:.3f} s, used {overlap_scale:.3f} s", flush=True)
    full_weight = np.clip(proximity_weight * overlap_weight, 0.01, 1.0).astype(np.float32)

    if variant == PRIMARY_VARIANT:
        selected["graph_relation"] = selected["relation_type"].map(normalize_flow_relation)
        selected["graph_weight"] = full_weight
    elif variant == ABLATION_VARIANT:
        selected["graph_relation"] = "flow_to_event"
        selected["graph_weight"] = np.ones(len(selected), dtype=np.float32)
    else:
        raise ValueError(f"Unknown graph variant: {variant!r}")

    for relation_name, group in selected.groupby("graph_relation", sort=True):
        add_relation(
            storage,
            str(relation_name),
            group["flow_node"].astype("int64").to_numpy(),
            group["event_node"].astype("int64").to_numpy(),
            group["graph_weight"].astype("float32").to_numpy(),
        )

    return GraphBundle(
        name=variant,
        relations=finalize_relations(storage),
        event_count=event_count,
        flow_count=flow_count,
    )


full_graph = build_graph_bundle(PRIMARY_VARIANT)
ablation_graph = build_graph_bundle(ABLATION_VARIANT)
graphs = {
    PRIMARY_VARIANT: full_graph,
    ABLATION_VARIANT: ablation_graph,
}
if INCLUDE_EVENT_ONLY_VARIANT:
    graphs[EVENT_ONLY_VARIANT] = build_graph_bundle(EVENT_ONLY_VARIANT)

relation_stat_rows: list[dict[str, object]] = []
for variant_name, bundle in graphs.items():
    for relation_name, edges in bundle.relations.items():
        relation_stat_rows.append(
            {
                "variant": variant_name,
                "relation": relation_name,
                "edge_count": len(edges.source),
                "weight_min": float(edges.weight.min()),
                "weight_mean": float(edges.weight.mean()),
                "weight_max": float(edges.weight.max()),
                "unique_source_nodes": int(np.unique(edges.source).size),
                "unique_destination_nodes": int(np.unique(edges.destination).size),
            }
        )
relation_statistics = pd.DataFrame(relation_stat_rows)
relation_statistics.to_csv(GRAPH_RELATIONS_OUT, index=False)
display(relation_statistics)

# Causality check: every event-to-event edge must point forward in sorted event order.
for variant_name, bundle in graphs.items():
    for relation_name, edges in bundle.relations.items():
        if relation_name == "event_self":
            continue  # v3: self-loops are not temporal edges
        event_edge_mask = (edges.source < event_count) & (edges.destination < event_count)
        if event_edge_mask.any() and not np.all(
            edges.source[event_edge_mask] < edges.destination[event_edge_mask]
        ):
            raise AssertionError(
                f"Non-causal event edge found in {variant_name}/{relation_name}."
            )

## 6. Define the relation-aware graph convolution and classifier

In [ ]:
def graph_to_device(
    bundle: GraphBundle,
    device: torch.device,
) -> dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
    output: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
    for relation_name, edges in bundle.relations.items():
        output[relation_name] = (
            torch.as_tensor(edges.source, dtype=torch.long, device=device),
            torch.as_tensor(edges.destination, dtype=torch.long, device=device),
            torch.as_tensor(edges.weight, dtype=torch.float32, device=device),
        )
    return output


class RelationalGraphConvolution(nn.Module):
    def __init__(self, hidden_dim: int, relation_names: Sequence[str], dropout: float) -> None:
        super().__init__()
        self.self_linear = nn.Linear(hidden_dim, hidden_dim)
        self.relation_names: tuple[str, ...] = tuple(relation_names)
        self.relation_linears: nn.ModuleDict = nn.ModuleDict(
            {
                relation_name: nn.Linear(hidden_dim, hidden_dim, bias=False)
                for relation_name in self.relation_names
            }
        )
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        node_state: torch.Tensor,
        relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    ) -> torch.Tensor:
        output = self.self_linear(node_state)
        node_count = node_state.shape[0]

        for relation_name in self.relation_names:
            linear = cast(nn.Linear, self.relation_linears[relation_name])
            source, destination, weight = relation_tensors[relation_name]
            if source.numel() == 0:
                continue
            messages = linear(node_state.index_select(0, source))
            messages = messages * weight.unsqueeze(1)
            aggregate = torch.zeros_like(output)
            aggregate.index_add_(0, destination, messages)
            degree = torch.zeros(node_count, dtype=node_state.dtype, device=node_state.device)
            degree.index_add_(0, destination, weight.abs())
            aggregate = aggregate / degree.clamp_min(1.0).unsqueeze(1)
            output = output + aggregate

        return self.layer_norm(self.dropout(output))


class RGCNEventClassifier(nn.Module):
    def __init__(
        self,
        event_input_dim: int,
        flow_input_dim: int,
        hidden_dim: int,
        relation_names: Sequence[str],
        layer_count: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.event_encoder = nn.Sequential(
            nn.Linear(event_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.flow_encoder = nn.Sequential(
            nn.Linear(flow_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.node_type_embedding = nn.Embedding(2, hidden_dim)
        self.layers: nn.ModuleList = nn.ModuleList(
            [
                RelationalGraphConvolution(hidden_dim, relation_names, dropout)
                for _ in range(layer_count)
            ]
        )
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(
        self,
        event_features: torch.Tensor,
        flow_features: torch.Tensor,
        relation_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    ) -> torch.Tensor:
        event_state = self.event_encoder(event_features)
        flow_state = self.flow_encoder(flow_features)
        node_state = torch.cat([event_state, flow_state], dim=0)

        type_ids = torch.cat(
            [
                torch.zeros(event_state.shape[0], dtype=torch.long, device=node_state.device),
                torch.ones(flow_state.shape[0], dtype=torch.long, device=node_state.device),
            ]
        )
        node_state = node_state + self.node_type_embedding(type_ids)

        for layer_index in range(len(self.layers)):
            layer = cast(RelationalGraphConvolution, self.layers[layer_index])
            residual = node_state
            node_state = self.activation(layer(node_state, relation_tensors))
            node_state = self.dropout(node_state + residual)

        return self.classifier(node_state[: event_state.shape[0]]).squeeze(1)


@dataclass(frozen=True)
class TrainingConfiguration:
    hidden_dim: int
    layer_count: int
    dropout: float
    learning_rate: float
    weight_decay: float
    maximum_epochs: int
    early_stopping_patience: int
    gradient_clip_norm: float


training_configuration = TrainingConfiguration(
    hidden_dim=HIDDEN_DIM,
    layer_count=N_LAYERS,
    dropout=DROPOUT,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    maximum_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
)

print(training_configuration)

## 7. Train the full and ablation R-GCN variants

In [ ]:
event_tensor = torch.as_tensor(event_matrix, dtype=torch.float32, device=DEVICE)
flow_tensor = torch.as_tensor(flow_matrix, dtype=torch.float32, device=DEVICE)
label_tensor = torch.as_tensor(
    model_data[TARGET_COLUMN].to_numpy(dtype=np.float32),
    dtype=torch.float32,
    device=DEVICE,
)

split_masks_np = {
    split_name: model_data[SPLIT_COLUMN].eq(split_name).to_numpy(dtype=bool)
    for split_name in ["train", "validation", "test"]
}
split_masks = {
    split_name: torch.as_tensor(mask, dtype=torch.bool, device=DEVICE)
    for split_name, mask in split_masks_np.items()
}


def predict_probabilities(
    model: RGCNEventClassifier,
    graph_tensors: Mapping[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        logits = model(event_tensor, flow_tensor, graph_tensors)
        probabilities = torch.sigmoid(logits)
    return probabilities.detach().cpu().numpy().astype(float)


def train_variant(
    variant_name: str,
    bundle: GraphBundle,
) -> tuple[
    RGCNEventClassifier,
    pd.DataFrame,
    pd.DataFrame,
    dict[str, object],
    dict[str, float | int],
]:
    seed_everything(RANDOM_STATE)
    relation_names = sorted(bundle.relations)
    graph_tensors = graph_to_device(bundle, DEVICE)
    model = RGCNEventClassifier(
        event_input_dim=event_matrix.shape[1],
        flow_input_dim=flow_matrix.shape[1],
        hidden_dim=HIDDEN_DIM,
        relation_names=relation_names,
        layer_count=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_labels = label_tensor[split_masks["train"]]
    positive_count = require_float(train_labels.sum().detach().cpu().item(), "training positives")
    negative_count = require_float(
        train_labels.numel() - positive_count,
        "training negatives",
    )
    if positive_count <= 0.0 or negative_count <= 0.0:
        raise ValueError("The training partition must contain both target classes.")
    positive_weight = torch.tensor(
        negative_count / positive_count,
        dtype=torch.float32,
        device=DEVICE,
    )
    loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history_rows: list[dict[str, object]] = []
    best_state: dict[str, torch.Tensor] | None = None
    best_score = -math.inf
    epochs_without_improvement = 0
    training_started = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(event_tensor, flow_tensor, graph_tensors)
        train_loss = loss_function(logits[split_masks["train"]], train_labels)
        train_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_logits = model(event_tensor, flow_tensor, graph_tensors)
            validation_loss = loss_function(
                validation_logits[split_masks["validation"]],
                label_tensor[split_masks["validation"]],
            )
            validation_probabilities = torch.sigmoid(
                validation_logits[split_masks["validation"]]
            ).detach().cpu().numpy()

        validation_labels = model_data.loc[
            split_masks_np["validation"], TARGET_COLUMN
        ].to_numpy(dtype=np.int8)
        validation_auprc = safe_auprc(validation_labels, validation_probabilities)
        monitoring_score = validation_auprc

        history_rows.append(
            {
                "variant": variant_name,
                "epoch": epoch,
                "train_loss": float(train_loss.detach().cpu().item()),
                "validation_loss": float(validation_loss.detach().cpu().item()),
                "validation_auprc": validation_auprc,
            }
        )

        if monitoring_score > best_score + 1e-8:
            best_score = monitoring_score
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"{variant_name}: epoch={epoch:03d} "
                f"train_loss={train_loss.item():.5f} "
                f"val_loss={validation_loss.item():.5f} "
                f"val_auprc={validation_auprc:.5f}"
            )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"{variant_name}: early stopping at epoch {epoch}.")
            break

    training_seconds = time.perf_counter() - training_started
    if best_state is None:
        raise RuntimeError(f"No best model state was recorded for {variant_name}.")
    model.load_state_dict(best_state)

    all_probabilities = predict_probabilities(model, graph_tensors)
    validation_labels = model_data.loc[
        split_masks_np["validation"], TARGET_COLUMN
    ].to_numpy(dtype=np.int8)
    validation_probabilities = all_probabilities[split_masks_np["validation"]]
    selected_threshold, validation_best_f1 = select_f1_threshold(
        validation_labels,
        validation_probabilities,
    )

    validation_metrics = classification_metrics(
        validation_labels,
        validation_probabilities,
        selected_threshold,
    )
    test_labels = model_data.loc[
        split_masks_np["test"], TARGET_COLUMN
    ].to_numpy(dtype=np.int8)
    test_probabilities = all_probabilities[split_masks_np["test"]]
    test_metrics = classification_metrics(
        test_labels,
        test_probabilities,
        selected_threshold,
    )

    prediction_frames: list[pd.DataFrame] = []
    for split_name in ["validation", "test"]:
        mask = split_masks_np[split_name]
        probabilities = all_probabilities[mask]
        prediction_frames.append(
            pd.DataFrame(
                {
                    "event_id": model_data.loc[mask, ID_COLUMN].astype(str).to_numpy(),
                    "event_time_utc": model_data.loc[mask, TIME_COLUMN].astype(str).to_numpy(),
                    "split": split_name,
                    "variant": variant_name,
                    "model_target": model_data.loc[mask, TARGET_COLUMN].to_numpy(dtype=np.int8),
                    "probability": probabilities,
                    "selected_threshold": selected_threshold,
                    "predicted_label": (probabilities >= selected_threshold).astype(np.int8),
                }
            )
        )
    predictions = pd.concat(prediction_frames, ignore_index=True)

    checkpoint_path = MODELS_DIR / f"{slugify(variant_name)}_rgcn.pt"
    torch.save(
        {
            "model_version": MODEL_VERSION,
            "variant": variant_name,
            "model_state_dict": model.state_dict(),
            "event_input_dim": event_matrix.shape[1],
            "flow_input_dim": flow_matrix.shape[1],
            "hidden_dim": HIDDEN_DIM,
            "layer_count": N_LAYERS,
            "dropout": DROPOUT,
            "relation_names": relation_names,
            "selected_threshold": selected_threshold,
            "target_definition": target_definition,
        },
        checkpoint_path,
    )

    threshold_record = {
        "variant": variant_name,
        "policy": "maximum_validation_f1",
        "selected_threshold": selected_threshold,
        "validation_f1_at_threshold": validation_best_f1,
    }
    result_record: dict[str, float | int] = {
        "epochs_completed": len(history_rows),
        "training_seconds": training_seconds,
        "selected_threshold": selected_threshold,
        "validation_auprc": float(validation_metrics["auprc"]),
        "validation_f1": float(validation_metrics["f1"]),
        "test_auprc": float(test_metrics["auprc"]),
        "test_roc_auc": float(test_metrics["roc_auc"]),
        "test_precision": float(test_metrics["precision"]),
        "test_recall": float(test_metrics["recall"]),
        "test_f1": float(test_metrics["f1"]),
        "test_balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "test_brier_score": float(test_metrics["brier_score"]),
        "test_tn": int(test_metrics["tn"]),
        "test_fp": int(test_metrics["fp"]),
        "test_fn": int(test_metrics["fn"]),
        "test_tp": int(test_metrics["tp"]),
    }
    return (
        model,
        pd.DataFrame(history_rows),
        predictions,
        threshold_record,
        result_record,
    )


trained_models: dict[str, RGCNEventClassifier] = {}
history_frames: list[pd.DataFrame] = []
prediction_frames: list[pd.DataFrame] = []
threshold_records: list[dict[str, object]] = []
result_rows: list[dict[str, object]] = []

for variant_name, bundle in graphs.items():
    model, history, predictions, threshold_record, result_record = train_variant(
        variant_name,
        bundle,
    )
    trained_models[variant_name] = model
    history_frames.append(history)
    prediction_frames.append(predictions)
    threshold_records.append(threshold_record)
    result_rows.append({"variant": variant_name, **result_record})

training_history = pd.concat(history_frames, ignore_index=True)
prediction_table = pd.concat(prediction_frames, ignore_index=True)
results = pd.DataFrame(result_rows)

training_history.to_csv(HISTORY_OUT, index=False)
prediction_table.to_csv(PREDICTIONS_OUT, index=False)
THRESHOLDS_OUT.write_text(
    json.dumps(threshold_records, indent=2, default=str),
    encoding="utf-8",
)

display(results)

## 8. Measure local and amortized inference latency

In [ ]:
def build_incoming_adjacency(
    bundle: GraphBundle,
) -> dict[int, list[tuple[str, int, float]]]:
    incoming: dict[int, list[tuple[str, int, float]]] = {}
    for relation_name, edges in bundle.relations.items():
        for source, destination, weight in zip(
            edges.source.tolist(),
            edges.destination.tolist(),
            edges.weight.tolist(),
        ):
            incoming.setdefault(int(destination), []).append(
                (relation_name, int(source), float(weight))
            )
    return incoming


def make_local_subgraph(
    target_event_index: int,
    bundle: GraphBundle,
    incoming: dict[int, list[tuple[str, int, float]]],
    depth: int,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    int,
]:
    selected_nodes: set[int] = {target_event_index}
    frontier: set[int] = {target_event_index}
    for _ in range(depth):
        next_frontier: set[int] = set()
        for destination in frontier:
            for _, source, _ in incoming.get(destination, []):
                if source not in selected_nodes:
                    selected_nodes.add(source)
                    next_frontier.add(source)
        frontier = next_frontier
        if not frontier:
            break

    local_event_global = sorted(node for node in selected_nodes if node < bundle.event_count)
    local_flow_global = sorted(node for node in selected_nodes if node >= bundle.event_count)
    ordered_global = local_event_global + local_flow_global
    remap = {global_index: local_index for local_index, global_index in enumerate(ordered_global)}

    relation_parts: dict[str, list[tuple[int, int, float]]] = {}
    selected_set = set(ordered_global)
    for relation_name, edges in bundle.relations.items():
        for source, destination, weight in zip(
            edges.source.tolist(),
            edges.destination.tolist(),
            edges.weight.tolist(),
        ):
            source_int = int(source)
            destination_int = int(destination)
            if source_int in selected_set and destination_int in selected_set:
                relation_parts.setdefault(relation_name, []).append(
                    (remap[source_int], remap[destination_int], float(weight))
                )

    relation_tensors: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
    for relation_name in sorted(bundle.relations):
        parts = relation_parts.get(relation_name, [])
        if parts:
            source = torch.tensor([part[0] for part in parts], dtype=torch.long, device=DEVICE)
            destination = torch.tensor([part[1] for part in parts], dtype=torch.long, device=DEVICE)
            weight = torch.tensor([part[2] for part in parts], dtype=torch.float32, device=DEVICE)
        else:
            source = torch.empty(0, dtype=torch.long, device=DEVICE)
            destination = torch.empty(0, dtype=torch.long, device=DEVICE)
            weight = torch.empty(0, dtype=torch.float32, device=DEVICE)
        relation_tensors[relation_name] = (source, destination, weight)

    local_event_features = torch.as_tensor(
        event_matrix[np.asarray(local_event_global, dtype=np.int64)],
        dtype=torch.float32,
        device=DEVICE,
    )
    local_flow_positions = np.asarray(
        [global_index - bundle.event_count for global_index in local_flow_global],
        dtype=np.int64,
    )
    local_flow_features = torch.as_tensor(
        flow_matrix[local_flow_positions],
        dtype=torch.float32,
        device=DEVICE,
    )
    target_local_event_index = local_event_global.index(target_event_index)
    return (
        local_event_features,
        local_flow_features,
        relation_tensors,
        target_local_event_index,
    )


def synchronize_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def measure_variant_latency(
    variant_name: str,
    model: RGCNEventClassifier,
    bundle: GraphBundle,
) -> tuple[pd.DataFrame, dict[str, float]]:
    model.eval()
    graph_tensors = graph_to_device(bundle, DEVICE)

    # Full-graph batch latency, reported both per run and amortized per event.
    full_run_ms: list[float] = []
    with torch.no_grad():
        for _ in range(3):
            _ = model(event_tensor, flow_tensor, graph_tensors)
        for _ in range(FULL_GRAPH_LATENCY_RUNS):
            synchronize_device()
            started = time.perf_counter()
            _ = model(event_tensor, flow_tensor, graph_tensors)
            synchronize_device()
            full_run_ms.append((time.perf_counter() - started) * 1000.0)

    # Prepared causal local subgraph latency for sampled test events.
    incoming = build_incoming_adjacency(bundle)
    test_indices = np.flatnonzero(split_masks_np["test"])
    rng = np.random.default_rng(RANDOM_STATE)
    if len(test_indices) > LOCAL_LATENCY_SAMPLE_MAX:
        test_indices = np.sort(
            rng.choice(test_indices, size=LOCAL_LATENCY_SAMPLE_MAX, replace=False)
        )
    prepared_subgraphs = [
        make_local_subgraph(int(index), bundle, incoming, N_LAYERS)
        for index in test_indices
    ]
    if not prepared_subgraphs:
        raise ValueError("No test events are available for latency measurement.")

    with torch.no_grad():
        for position in range(min(LOCAL_LATENCY_WARMUP_RUNS, len(prepared_subgraphs))):
            local_event_x, local_flow_x, local_relations, target_local = prepared_subgraphs[position]
            _ = model(local_event_x, local_flow_x, local_relations)[target_local]

    local_rows: list[dict[str, object]] = []
    with torch.no_grad():
        for event_index, prepared in zip(test_indices.tolist(), prepared_subgraphs):
            local_event_x, local_flow_x, local_relations, target_local = prepared
            synchronize_device()
            started = time.perf_counter()
            probability = torch.sigmoid(
                model(local_event_x, local_flow_x, local_relations)[target_local]
            )
            synchronize_device()
            elapsed_ms = (time.perf_counter() - started) * 1000.0
            local_rows.append(
                {
                    "variant": variant_name,
                    "event_id": str(model_data.iloc[event_index][ID_COLUMN]),
                    "latency_type": "prepared_local_causal_subgraph",
                    "latency_ms": elapsed_ms,
                    "local_event_nodes": local_event_x.shape[0],
                    "local_flow_nodes": local_flow_x.shape[0],
                    "probability": float(probability.detach().cpu().item()),
                }
            )

    local_latency = np.asarray([row["latency_ms"] for row in local_rows], dtype=float)
    summary = {
        "local_latency_p50_ms": float(np.percentile(local_latency, 50)),
        "local_latency_mean_ms": float(np.mean(local_latency)),
        "local_latency_p95_ms": float(np.percentile(local_latency, 95)),
        "local_latency_samples": float(len(local_latency)),
        "full_graph_latency_p50_ms": float(np.percentile(full_run_ms, 50)),
        "full_graph_latency_p95_ms": float(np.percentile(full_run_ms, 95)),
        "full_graph_amortized_p95_ms_per_event": float(
            np.percentile(full_run_ms, 95) / max(bundle.event_count, 1)
        ),
    }
    return pd.DataFrame(local_rows), summary


latency_frames: list[pd.DataFrame] = []
latency_summaries: dict[str, dict[str, float]] = {}
for variant_name, model in trained_models.items():
    latency_frame, latency_summary = measure_variant_latency(
        variant_name,
        model,
        graphs[variant_name],
    )
    latency_frames.append(latency_frame)
    latency_summaries[variant_name] = latency_summary

latency_table = pd.concat(latency_frames, ignore_index=True)
latency_table.to_csv(LATENCY_OUT, index=False)

for variant_name, summary_values in latency_summaries.items():
    result_mask = results["variant"].eq(variant_name)
    for metric_name, metric_value in summary_values.items():
        results.loc[result_mask, metric_name] = metric_value

results.to_csv(RESULTS_OUT, index=False)
display(results)

## 9. Diagnostic feature attribution and learned relation norms

In [ ]:
primary_model = trained_models[PRIMARY_VARIANT]
primary_graph_tensors = graph_to_device(full_graph, DEVICE)
primary_model.eval()

# Gradient-times-input diagnostic attribution for the full R-GCN.
test_indices = np.flatnonzero(split_masks_np["test"])
rng = np.random.default_rng(RANDOM_STATE)
if len(test_indices) > ATTRIBUTION_SAMPLE_MAX:
    attribution_indices = np.sort(
        rng.choice(test_indices, size=ATTRIBUTION_SAMPLE_MAX, replace=False)
    )
else:
    attribution_indices = test_indices

attribution_event_x = event_tensor.detach().clone().requires_grad_(True)
attribution_flow_x = flow_tensor.detach().clone().requires_grad_(True)
primary_model.zero_grad(set_to_none=True)
attribution_logits = primary_model(
    attribution_event_x,
    attribution_flow_x,
    primary_graph_tensors,
)
attribution_score = attribution_logits[
    torch.as_tensor(attribution_indices, dtype=torch.long, device=DEVICE)
].sum()
attribution_score.backward()

if attribution_event_x.grad is None or attribution_flow_x.grad is None:
    raise RuntimeError("Gradient attribution did not produce input gradients.")

event_importance = (
    attribution_event_x.grad.detach() * attribution_event_x.detach()
).abs().mean(dim=0).cpu().numpy()
flow_importance = (
    attribution_flow_x.grad.detach() * attribution_flow_x.detach()
).abs().mean(dim=0).cpu().numpy()

feature_attribution = pd.concat(
    [
        pd.DataFrame(
            {
                "node_type": "event",
                "feature": event_feature_names,
                "mean_absolute_gradient_times_input": event_importance,
            }
        ),
        pd.DataFrame(
            {
                "node_type": "flow",
                "feature": flow_feature_names,
                "mean_absolute_gradient_times_input": flow_importance,
            }
        ),
    ],
    ignore_index=True,
)
feature_attribution["rank_within_node_type"] = (
    feature_attribution.groupby("node_type")["mean_absolute_gradient_times_input"]
    .rank(method="first", ascending=False)
    .astype("int64")
)
feature_attribution = feature_attribution.sort_values(
    ["node_type", "rank_within_node_type"], kind="mergesort"
)
feature_attribution.to_csv(FEATURE_ATTRIBUTION_OUT, index=False)

relation_norm_rows: list[dict[str, object]] = []
relation_module_pattern = re.compile(
    r"^layers\.(?P<layer_index>\d+)\.relation_linears\.(?P<relation_name>.+)$"
)

for variant_name, model in trained_models.items():
    for module_name, module in model.named_modules():
        match = relation_module_pattern.fullmatch(module_name)
        if match is None or not isinstance(module, nn.Linear):
            continue

        layer_index = require_int(
            match.group("layer_index"),
            "relation layer index",
        )
        relation_name = require_text(
            match.group("relation_name"),
            "relation name",
        )
        weight_norm = require_float(
            torch.linalg.vector_norm(module.weight.detach()).cpu().item(),
            "relation weight Frobenius norm",
        )

        relation_norm_rows.append(
            {
                "variant": variant_name,
                "layer_index": layer_index,
                "relation": relation_name,
                "weight_frobenius_norm": weight_norm,
            }
        )

relation_weight_norms = pd.DataFrame(relation_norm_rows)
relation_weight_norms.to_csv(RELATION_NORMS_OUT, index=False)

display(feature_attribution.groupby("node_type", group_keys=False).head(15))
display(relation_weight_norms)

## 10. Compare with the frozen Transformer and evaluate H2/H3

In [ ]:
full_result = results.loc[results["variant"].eq(PRIMARY_VARIANT)].iloc[0]
ablation_result = results.loc[results["variant"].eq(ABLATION_VARIANT)].iloc[0]

full_test_auprc = require_float(full_result["test_auprc"], "full R-GCN test AUPRC")
ablation_test_auprc = require_float(
    ablation_result["test_auprc"],
    "ablation test AUPRC",
)
full_p95_latency_ms = require_float(
    full_result["local_latency_p95_ms"],
    "full R-GCN local p95 latency",
)

h2_margin = full_test_auprc - transformer_test_auprc
h3_gain = full_test_auprc - ablation_test_auprc
h2_pass = h2_margin >= H2_REQUIRED_AUPRC_MARGIN
h3_performance_pass = h3_gain >= H3_REQUIRED_AUPRC_GAIN
h3_latency_pass = full_p95_latency_ms <= H3_MAX_P95_LATENCY_MS
h3_pass = h3_performance_pass and h3_latency_pass

comparison = pd.DataFrame(
    [
        {
            "comparison": "R-GCN versus temporal Transformer",
            "left_model": PRIMARY_VARIANT,
            "left_test_auprc": full_test_auprc,
            "right_model": "causal_temporal_transformer",
            "right_test_auprc": transformer_test_auprc,
            "auprc_difference": h2_margin,
        },
        {
            "comparison": "Lag/relation-aware R-GCN versus ablation",
            "left_model": PRIMARY_VARIANT,
            "left_test_auprc": full_test_auprc,
            "right_model": ABLATION_VARIANT,
            "right_test_auprc": ablation_test_auprc,
            "auprc_difference": h3_gain,
        },
    ]
)
comparison.to_csv(COMPARISON_OUT, index=False)

hypothesis_evaluation = pd.DataFrame(
    [
        {
            "hypothesis": "H2",
            "criterion": "R-GCN test AUPRC - Transformer test AUPRC >= 0.03",
            "observed_value": h2_margin,
            "required_value": H2_REQUIRED_AUPRC_MARGIN,
            "latency_value_ms": np.nan,
            "latency_limit_ms": np.nan,
            "supported": h2_pass,
        },
        {
            "hypothesis": "H3",
            "criterion": (
                "Full R-GCN test AUPRC - lag/relation ablation test AUPRC >= 0.05 "
                "and full local p95 latency <= 500 ms"
            ),
            "observed_value": h3_gain,
            "required_value": H3_REQUIRED_AUPRC_GAIN,
            "latency_value_ms": full_p95_latency_ms,
            "latency_limit_ms": H3_MAX_P95_LATENCY_MS,
            "supported": h3_pass,
        },
    ]
)
hypothesis_evaluation.to_csv(HYPOTHESIS_OUT, index=False)

display(comparison)
display(hypothesis_evaluation)

print(f"H2 AUPRC margin: {h2_margin:+.6f} -> {'SUPPORTED' if h2_pass else 'NOT SUPPORTED'}")
print(f"H3 AUPRC gain:   {h3_gain:+.6f}")
print(f"H3 p95 latency:  {full_p95_latency_ms:.3f} ms")
print(f"H3 result:       {'SUPPORTED' if h3_pass else 'NOT SUPPORTED'}")

## 11. Generate figures

In [ ]:
for variant_name in results["variant"].astype(str).tolist():
    variant_history = training_history.loc[training_history["variant"].eq(variant_name)]
    plt.figure(figsize=(8, 5))
    plt.plot(variant_history["epoch"], variant_history["train_loss"], label="Training loss")
    plt.plot(
        variant_history["epoch"],
        variant_history["validation_loss"],
        label="Validation loss",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"R-GCN training history: {variant_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{slugify(variant_name)}_training_history.png", dpi=180)
    plt.close()

    test_predictions = prediction_table.loc[
        prediction_table["variant"].eq(variant_name)
        & prediction_table["split"].eq("test")
    ]
    precision, recall, _ = precision_recall_curve(
        test_predictions["model_target"].to_numpy(dtype=np.int8),
        test_predictions["probability"].to_numpy(dtype=float),
    )
    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Test precision-recall curve: {variant_name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{slugify(variant_name)}_test_precision_recall.png", dpi=180)
    plt.close()

    matrix = confusion_matrix(
        test_predictions["model_target"].to_numpy(dtype=np.int8),
        test_predictions["predicted_label"].to_numpy(dtype=np.int8),
        labels=[0, 1],
    )
    plt.figure(figsize=(5, 5))
    plt.imshow(matrix)
    plt.xticks([0, 1], ["Predicted 0", "Predicted 1"])
    plt.yticks([0, 1], ["Actual 0", "Actual 1"])
    for row_index in range(2):
        for column_index in range(2):
            plt.text(column_index, row_index, str(matrix[row_index, column_index]), ha="center", va="center")
    plt.title(f"Test confusion matrix: {variant_name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{slugify(variant_name)}_test_confusion_matrix.png", dpi=180)
    plt.close()

for node_type in ["event", "flow"]:
    top_features = (
        feature_attribution.loc[feature_attribution["node_type"].eq(node_type)]
        .nsmallest(20, "rank_within_node_type")
        .sort_values("mean_absolute_gradient_times_input")
    )
    if not top_features.empty:
        plt.figure(figsize=(9, 7))
        plt.barh(
            top_features["feature"],
            top_features["mean_absolute_gradient_times_input"],
        )
        plt.xlabel("Mean |gradient × input|")
        plt.title(f"Full R-GCN {node_type}-feature attribution")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"full_rgcn_{node_type}_feature_attribution.png", dpi=180)
        plt.close()

full_relation_norms = relation_weight_norms.loc[
    relation_weight_norms["variant"].eq(PRIMARY_VARIANT)
]
if not full_relation_norms.empty:
    pivot = full_relation_norms.pivot(
        index="relation",
        columns="layer_index",
        values="weight_frobenius_norm",
    )
    pivot.plot(kind="barh", figsize=(10, 7))
    plt.xlabel("Frobenius norm")
    plt.title("Full R-GCN learned relation weight norms")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "full_rgcn_relation_weight_norms.png", dpi=180)
    plt.close()

## 12. Freeze graph/model references, metadata, checksums, and regression checks

In [ ]:
full_result = results.loc[results["variant"].eq(PRIMARY_VARIANT)].iloc[0]
ablation_result = results.loc[results["variant"].eq(ABLATION_VARIANT)].iloc[0]

rgcn_reference = {
    "model_version": MODEL_VERSION,
    "model": "causal_heterogeneous_rgcn",
    "primary_variant": PRIMARY_VARIANT,
    "target_definition": target_definition,
    "selected_threshold": optional_float(full_result["selected_threshold"]),
    "validation_auprc": optional_float(full_result["validation_auprc"]),
    "test_auprc": optional_float(full_result["test_auprc"]),
    "test_f1": optional_float(full_result["test_f1"]),
    "test_recall": optional_float(full_result["test_recall"]),
    "local_latency_p95_ms": optional_float(full_result["local_latency_p95_ms"]),
    "transformer_test_auprc": transformer_test_auprc,
    "h2_auprc_margin": h2_margin,
    "h2_supported": h2_pass,
    "ablation_test_auprc": optional_float(ablation_result["test_auprc"]),
    "h3_auprc_gain": h3_gain,
    "h3_latency_pass": h3_latency_pass,
    "h3_supported": h3_pass,
    "interpretation": (
        "Controlled cross-domain alignment performance; not evidence of a site-coincident "
        "or causal aircraft-cyber relationship."
    ),
}
REFERENCE_OUT.write_text(json.dumps(rgcn_reference, indent=2, default=str), encoding="utf-8")

configuration_payload = {
    "model_version": MODEL_VERSION,
    "random_state": RANDOM_STATE,
    "primary_variant": PRIMARY_VARIANT,
    "ablation_variant": ABLATION_VARIANT,
    "training_configuration": asdict(training_configuration),
    "device": str(DEVICE),
    "graph_policy": {
        "event_nodes": event_count,
        "selected_flow_nodes": flow_count,
        "maximum_flows_per_event": MAX_FLOWS_PER_EVENT,
        "selection_uses_labels": False,
        "flow_to_event_only": True,
        "event_to_flow_edges": False,
        "event_event_edges_are_causal": True,
        "full_flow_relations": [
            "flow_pre_event_to_event",
            "flow_overlaps_event_to_event",
            "flow_post_event_to_event",
            "flow_other_to_event",
        ],
        "lag_weight_scale_s": LAG_WEIGHT_SCALE_S,
    },
    "hypothesis_criteria": {
        "H2_required_auprc_margin": H2_REQUIRED_AUPRC_MARGIN,
        "H3_required_auprc_gain": H3_REQUIRED_AUPRC_GAIN,
        "H3_max_p95_latency_ms": H3_MAX_P95_LATENCY_MS,
    },
}
CONFIG_OUT.write_text(json.dumps(configuration_payload, indent=2, default=str), encoding="utf-8")

graph_manifest = {
    "model_version": MODEL_VERSION,
    "node_counts": {
        "event": event_count,
        "flow": flow_count,
        "total": event_count + flow_count,
    },
    "selected_event_flow_edges": len(selected_edges),
    "events_with_selected_flows": edge_event_count,
    "relations": {
        variant_name: {
            relation_name: len(edges.source)
            for relation_name, edges in bundle.relations.items()
        }
        for variant_name, bundle in graphs.items()
    },
    "event_features": {
        "raw_numeric": event_numeric,
        "raw_categorical": event_categorical,
        "dropped_training_constant_or_empty": dropped_event_features,
        "transformed_count": event_matrix.shape[1],
    },
    "flow_features": {
        "raw_numeric": flow_numeric,
        "raw_categorical": flow_categorical,
        "dropped_training_constant_or_empty": dropped_flow_features,
        "transformed_count": flow_matrix.shape[1],
    },
}
GRAPH_MANIFEST_OUT.write_text(json.dumps(graph_manifest, indent=2, default=str), encoding="utf-8")

input_checksums = {
    "feature_bank": sha256_file(FEATURES_IN),
    "event_flow_edges": sha256_file(EDGES_IN),
    "split_assignments": sha256_file(SPLITS_IN),
    "feature_manifest": sha256_file(FEATURE_MANIFEST_IN),
    "target_definition": sha256_file(TARGET_DEFINITION_IN),
    "transformer_reference": sha256_file(TRANSFORMER_REFERENCE_IN),
}

regression_checks = [
    {
        "metric": "feature_rows_equal_split_rows",
        "expected": len(features),
        "actual": len(model_data),
        "match": len(features) == len(model_data),
    },
    {
        "metric": "selected_edges_obey_per_event_bound",
        "expected": True,
        "actual": bool(selected_edges.groupby("event_id").size().max() <= MAX_FLOWS_PER_EVENT),
        "match": bool(selected_edges.groupby("event_id").size().max() <= MAX_FLOWS_PER_EVENT),
    },
    {
        "metric": "selected_edge_table_contains_no_label_columns",
        "expected": 0,
        "actual": len(
            {
                "Label",
                "Attack",
                "nf3_binary_label",
                "nf3_attack_category",
                "cyber_attack_window_label",
            }
            & set(selected_edges.columns)
        ),
        "match": len(
            {
                "Label",
                "Attack",
                "nf3_binary_label",
                "nf3_attack_category",
                "cyber_attack_window_label",
            }
            & set(selected_edges.columns)
        )
        == 0,
    },
    {
        "metric": "all_event_edges_are_causal",
        "expected": True,
        "actual": all(
            np.all(edges.source[mask] < edges.destination[mask])
            for bundle in graphs.values()
            for relation_name, edges in bundle.relations.items()
            if relation_name != "event_self"  # v3: self-loops are not temporal edges
            for mask in [(edges.source < event_count) & (edges.destination < event_count)]
        ),
        "match": all(
            np.all(edges.source[mask] < edges.destination[mask])
            for bundle in graphs.values()
            for relation_name, edges in bundle.relations.items()
            if relation_name != "event_self"
            for mask in [(edges.source < event_count) & (edges.destination < event_count)]
        ),
    },
    {
        "metric": "all_rgcn_variants_completed",
        "expected": len(graphs),  # v3: 2 variants, or 3 with the event-only self-loop variant
        "actual": len(results),
        "match": len(results) == len(graphs),
    },
    {
        "metric": "test_prediction_rows_for_each_variant",
        "expected": require_int(model_data[SPLIT_COLUMN].eq("test").sum(), "test rows") * len(graphs),
        "actual": require_int(prediction_table["split"].eq("test").sum(), "test predictions"),
        "match": require_int(prediction_table["split"].eq("test").sum(), "test predictions")
        == require_int(model_data[SPLIT_COLUMN].eq("test").sum(), "test rows") * len(graphs),
    },
    {
        "metric": "full_local_p95_latency_under_500ms",
        "expected": True,
        "actual": h3_latency_pass,
        "match": h3_latency_pass,
    },
]

summary = {
    "definitions": configuration_payload,
    "interpretation": {
        "site_coincident_claim": False,
        "causal_claim": False,
        "statement": (
            "The R-GCN uses controlled point-event/NF3 pairings and causal graph direction. "
            "It does not establish real simultaneous PANC aircraft and network events."
        ),
    },
    "inputs": {
        "event_rows": len(model_data),
        "unique_aircraft_keys": int(model_data[GROUP_COLUMN].nunique()),
        "event_time_start_utc": model_data[TIME_COLUMN].min().isoformat(),
        "event_time_end_utc": model_data[TIME_COLUMN].max().isoformat(),
        "split_counts": model_data[SPLIT_COLUMN].value_counts().to_dict(),
        "positive_targets": int(model_data[TARGET_COLUMN].sum()),
        "target_prevalence": float(model_data[TARGET_COLUMN].mean()),
        "input_sha256": input_checksums,
    },
    "graph": graph_manifest,
    "outputs": {
        "model_runs": len(results),
        "prediction_rows": len(prediction_table),
        "history_rows": len(training_history),
        "feature_attribution_rows": len(feature_attribution),
        "relation_norm_rows": len(relation_weight_norms),
        "saved_model_files": len(list(MODELS_DIR.glob("*.pt"))),
        "saved_preprocessor_files": len(list(PREPROCESSING_DIR.glob("*.joblib"))),
        "saved_figure_files": len(list(FIGURES_DIR.glob("*.png"))),
        "rgcn_reference": rgcn_reference,
    },
    "environment": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "device": str(DEVICE),
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "regression_checks": regression_checks,
}
SUMMARY_OUT.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

artifact_descriptions = {
    SELECTED_EDGES_OUT.name: "Label-blind, bounded event-to-flow edge sample used by the R-GCN.",
    NODE_MANIFEST_OUT.name: "Stable graph node index for point-event and selected flow nodes.",
    GRAPH_RELATIONS_OUT.name: "Per-variant relation counts, weight statistics, and node coverage.",
    RESULTS_OUT.name: "Validation/test metrics, thresholds, training time, and latency for both R-GCN variants.",
    PREDICTIONS_OUT.name: "Event-level validation and test probabilities for both R-GCN variants.",
    THRESHOLDS_OUT.name: "Validation F1 threshold-selection results.",
    HISTORY_OUT.name: "Per-epoch training loss, validation loss, and validation AUPRC.",
    LATENCY_OUT.name: "Prepared local causal-subgraph inference latency samples.",
    RELATION_NORMS_OUT.name: "Learned relation-transformation Frobenius norms by layer.",
    FEATURE_ATTRIBUTION_OUT.name: "Gradient-times-input diagnostic feature attribution.",
    COMPARISON_OUT.name: "R-GCN comparisons with the frozen Transformer and lag/relation ablation.",
    HYPOTHESIS_OUT.name: "Explicit H2 and H3 decision table.",
    GRAPH_MANIFEST_OUT.name: "Frozen node, edge, relation, and feature construction policy.",
    REFERENCE_OUT.name: "Frozen R-GCN performance reference.",
    CONFIG_OUT.name: "Frozen architecture, graph policy, and hypothesis criteria.",
    SUMMARY_OUT.name: "Build metadata, environment, provenance, and regression checks.",
}
data_dictionary = pd.DataFrame(
    [
        {"artifact": artifact, "artifact_role": description}
        for artifact, description in artifact_descriptions.items()
    ]
)
data_dictionary.to_csv(DATA_DICTIONARY_OUT, index=False)

artifact_paths = [
    SELECTED_EDGES_OUT,
    NODE_MANIFEST_OUT,
    GRAPH_RELATIONS_OUT,
    RESULTS_OUT,
    PREDICTIONS_OUT,
    THRESHOLDS_OUT,
    HISTORY_OUT,
    LATENCY_OUT,
    RELATION_NORMS_OUT,
    FEATURE_ATTRIBUTION_OUT,
    COMPARISON_OUT,
    HYPOTHESIS_OUT,
    GRAPH_MANIFEST_OUT,
    REFERENCE_OUT,
    CONFIG_OUT,
    SUMMARY_OUT,
    DATA_DICTIONARY_OUT,
]
artifact_paths.extend(sorted(MODELS_DIR.glob("*.pt")))
artifact_paths.extend(sorted(PREPROCESSING_DIR.glob("*.joblib")))
artifact_paths.extend(sorted(FIGURES_DIR.glob("*.png")))

checksum_lines: list[str] = []
for artifact_path in artifact_paths:
    if artifact_path.is_file() and artifact_path != CHECKSUMS_OUT:
        checksum_lines.append(
            f"{sha256_file(artifact_path)}  {artifact_path.relative_to(OUTPUT_DIR).as_posix()}"
        )
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

final_checks = pd.DataFrame(regression_checks)
display(final_checks)
structural_checks = final_checks.loc[
    ~final_checks["metric"].eq("full_local_p95_latency_under_500ms")
]
if not structural_checks["match"].all():
    raise AssertionError("One or more structural R-GCN regression checks failed.")
if not h3_latency_pass:
    warnings.warn(
        "The full R-GCN local p95 inference latency exceeded 500 ms on this environment. "
        "The result was retained for transparent H3 evaluation.",
        RuntimeWarning,
    )

print("\nNotebook 06 completed successfully.")
print(f"Full R-GCN test AUPRC:       {full_test_auprc:.6f}")
print(f"Transformer test AUPRC:      {transformer_test_auprc:.6f}")
print(f"H2 AUPRC margin:             {h2_margin:+.6f}")
print(f"Lag/relation ablation AUPRC: {ablation_test_auprc:.6f}")
print(f"H3 AUPRC gain:               {h3_gain:+.6f}")
print(f"Full local p95 latency:      {full_p95_latency_ms:.3f} ms")
print(f"Output directory:            {OUTPUT_DIR}")

## Output interpretation

Use `rgcn_hypothesis_evaluation_v2.csv` as the primary H2/H3 decision table. A hypothesis that is not supported should be reported transparently rather than prompting post-hoc threshold or graph changes.

The next stage should consolidate tabular, Transformer, and R-GCN results; run planned robustness checks; and prepare final figures/tables for the Praxis manuscript.